# 17.2 整數格子如何代表0.7這樣的小數？


想用整數保存0.7與1.2，先決定相鄰整數代表的浮點間隔，叫**scale（刻度）**。若scale0.5，碼1代表0.5、碼2代表1.0。0.7除0.5得1.4，取最近整數1；還原乘0.5得到0.5，差0.2。

把原值映到整數格叫量化；按刻度轉回近似值叫**反量化（dequantization）**。原值在格子附近的偏移沒有保存，所以不能完整找回0.7。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-17-grid-mapping.svg")))

In [ ]:
import torch

x = torch.tensor([-0.7, 0.2, 0.7, 1.2])
scale = 0.5
q = (x / scale).round().clamp(-127, 127).to(torch.int8)
restored = q.float() * scale
print("整數格", q.tolist())
print("還原值", restored.tolist())
print("差值", (x - restored).round(decimals=4).tolist())

輸入四個固定數，`round()`逐格取最近整數，`clamp(-127,127)`把超出碼範圍的值壓到端點，最後`to(int8)`才真的每格存一byte。本例選正負對稱−127至127，留下int8可存的−128不用。

碼為`[-1,0,1,2]`；轉回浮點再乘scale，得`[-0.5,0,0.5,1]`；原值減還原值約為`[-0.2,0.2,0.2,0.2]`。0.2離0較近，落到0是這個刻度的結果。沒有截斷時最近格誤差不超過半格，超過範圍後則可能更大。剛好在兩格中間，PyTorch取最近偶數。

浮點打印可能出現0.20000000298這類尾數；`tensor.round(...).tolist()`沒有指定顯示的小數位，仍會顯示有限精度保存值。這種小尾差與本例約0.2的量化誤差不同。

練習只把scale改0.25，預測碼`[-3,1,3,5]`、還原`[-0.75,0.25,0.75,1.25]`，誤差約0.05。但同樣127個正格只覆蓋到31.75，刻度越細，可覆蓋範圍也越小。
